In [ ]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# print(df.head())

In [ ]:
text = df["text"]
sentiment = df["sentiment"]

In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_val, sentiment_train, sentiment_val = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from models.preprocess import tokenizer
from scipy.sparse import hstack

word_vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    token_pattern=None,
    ngram_range=(1,2),
    max_features=40000,
    # min_df=3,
    # sublinear_tf=True
    )

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=20000
)

word_train_tfidf = word_vectorizer.fit_transform(text_train)
word_val_tfidf = word_vectorizer.transform(text_val)

char_train_tfidf = char_vectorizer.fit_transform(text_train)
char_val_tfidf = char_vectorizer.transform(text_val)

text_train_tfidf = hstack([word_train_tfidf, char_train_tfidf])
text_val_tfidf = hstack([word_val_tfidf, char_val_tfidf])

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC(C=2, class_weight="balanced")
model.fit(text_train_tfidf, sentiment_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report # type: ignore

# prediction on training data
train_pred = model.predict(text_train_tfidf)

# prediction on test data
test_pred = model.predict(text_val_tfidf)

train_acc = accuracy_score(sentiment_train, train_pred)
test_acc = accuracy_score(sentiment_val, test_pred)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)
print("Test")
print(classification_report(sentiment_val, test_pred))

In [ ]:
import pickle

with open("../../models/SVM/sentiment_model.pkl", "wb") as f:
    pickle.dump((word_vectorizer, char_vectorizer, model), f)

In [ ]:
import pickle

with open("sentiment_model.pkl", "rb") as f:
    vectorizer, model = pickle.load(f)

test_text = "ไป ตาย ไป"

vec = vectorizer.transform([test_text])
prediction = model.predict(vec)

print("Prediction:", prediction[0])